# Phase 1.3: Cross-City User Identification

Identify users who have reviewed businesses in multiple cities.

In [1]:
import sys
sys.path.insert(0, '../../')

import pandas as pd
from config import *
from src.data.cache import DataCache

cache = DataCache(str(CACHE_DIR))
businesses = cache.load_dataframe('businesses_major_cities')
reviews = cache.load_dataframe('reviews_major_cities')
major_cities_meta = cache.load_metadata('major_cities')
major_cities = major_cities_meta['major_cities']

In [2]:
# Identify users with reviews in multiple cities
merged = reviews.merge(businesses[['business_id', 'city']], on='business_id')

user_city_counts = merged.groupby('user_id')['city'].nunique()
multi_city_users = user_city_counts[user_city_counts >= PHASE1_MIN_USER_CITIES].index.tolist()

print(f"Users with reviews in {PHASE1_MIN_USER_CITIES}+ cities: {len(multi_city_users):,}")
print(f"City distribution:")
print(user_city_counts.value_counts().head(10))

Users with reviews in 2+ cities: 1,376
City distribution:
city
1    47316
2     1263
3       90
4       19
5        3
7        1
Name: count, dtype: int64


In [3]:
# Further filter: minimum reviews per user
multi_city_reviews = reviews[reviews['user_id'].isin(multi_city_users)]
user_review_counts = multi_city_reviews['user_id'].value_counts()
active_users = user_review_counts[user_review_counts >= PHASE1_MIN_USER_REVIEWS].index.tolist()

print(f"Active multi-city users (≥{PHASE1_MIN_USER_REVIEWS} reviews): {len(active_users):,}")

# Final filtered reviews
reviews_final = reviews[reviews['user_id'].isin(active_users)]
print(f"Final review count: {len(reviews_final):,}")

Active multi-city users (≥10 reviews): 23
Final review count: 298


In [4]:
# Save final data
cache.save_dataframe(reviews_final, 'reviews_cross_city')
cache.save_metadata({'num_users': len(active_users)}, 'cross_city_stats')

print("✓ Cross-city user data cached")

✓ Cross-city user data cached
